In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import fastf1 as f1
from sklearn.linear_model import LinearRegression, RANSACRegressor

# Import local class (classes/F1Analyzer.py)
sys.path.append(str(Path('classes').resolve()))
from F1Analyzer import F1Analyzer

f1.set_log_level('WARNING')
f1.Cache.enable_cache('cache_strategy')

CIRCUIT_STATS = {
    'Bahrain Grand Prix': {'abrasivity': 5, 'lateral_energy': 3},
    'Saudi Arabian Grand Prix': {'abrasivity': 2, 'lateral_energy': 4},
    'Australian Grand Prix': {'abrasivity': 3, 'lateral_energy': 3},
    'Japanese Grand Prix': {'abrasivity': 5, 'lateral_energy': 5},
    'Chinese Grand Prix': {'abrasivity': 3, 'lateral_energy': 4},
    'Miami Grand Prix': {'abrasivity': 3, 'lateral_energy': 3},
    'Emilia Romagna Grand Prix': {'abrasivity': 3, 'lateral_energy': 3},
    'Monaco Grand Prix': {'abrasivity': 1, 'lateral_energy': 1},
    'Canadian Grand Prix': {'abrasivity': 2, 'lateral_energy': 2},
    'Spanish Grand Prix': {'abrasivity': 4, 'lateral_energy': 5},
    'Austrian Grand Prix': {'abrasivity': 2, 'lateral_energy': 3},
    'British Grand Prix': {'abrasivity': 3, 'lateral_energy': 5},
    'Hungarian Grand Prix': {'abrasivity': 2, 'lateral_energy': 3},
    'Belgian Grand Prix': {'abrasivity': 4, 'lateral_energy': 5},
    'Dutch Grand Prix': {'abrasivity': 3, 'lateral_energy': 5},
    'Italian Grand Prix': {'abrasivity': 2, 'lateral_energy': 3},
    'Azerbaijan Grand Prix': {'abrasivity': 2, 'lateral_energy': 2},
    'Singapore Grand Prix': {'abrasivity': 4, 'lateral_energy': 2},
    'United States Grand Prix': {'abrasivity': 4, 'lateral_energy': 4},
    'Mexico City Grand Prix': {'abrasivity': 2, 'lateral_energy': 2},
    'Sao Paulo Grand Prix': {'abrasivity': 3, 'lateral_energy': 3},
    'Las Vegas Grand Prix': {'abrasivity': 2, 'lateral_energy': 2},
    'Qatar Grand Prix': {'abrasivity': 4, 'lateral_energy': 5},
    'Abu Dhabi Grand Prix': {'abrasivity': 3, 'lateral_energy': 3},
}

COMPOUND_MAP = {'SOFT': 0, 'MEDIUM': 1, 'HARD': 2}


def _cache_event_folder_to_name(folder_name):
    part = folder_name.split('_', 1)[1]
    return part.replace('_', ' ')


def get_cached_race_events(cache_root='cache_strategy', year=2024):
    root = Path(cache_root) / str(year)
    if not root.exists():
        return []

    events = []
    for event_dir in sorted([d for d in root.iterdir() if d.is_dir()]):
        has_race = any(child.is_dir() and child.name.endswith('_Race') for child in event_dir.iterdir())
        if has_race:
            events.append(_cache_event_folder_to_name(event_dir.name))

    events = [e for e in events if e != 'United States Grand Prix']
    return events


def build_event_driver_laps(year, event_name, session_type='R'):
    analyzer = F1Analyzer(year, event_name, session_type, use_fuel_logic=True)
    session = analyzer.session

    all_rows = []

    if hasattr(session, 'total_laps') and session.total_laps is not None:
        race_max_laps = int(session.total_laps)
    else:
        race_max_laps = int(pd.to_numeric(session.laps['LapNumber'], errors='coerce').max())

    pirelli = CIRCUIT_STATS.get(event_name, {})
    abrasivity = pirelli.get('abrasivity', np.nan)
    lateral_energy = pirelli.get('lateral_energy', np.nan)

    for drv in session.drivers:
        laps = analyzer.get_clean_laps(drv)
        if laps.empty:
            continue

        laps['Compound'] = laps['Compound'].astype(str).str.upper()
        laps = laps[laps['Compound'].isin(COMPOUND_MAP.keys())].copy()
        if laps.empty:
            continue

        laps['LapNumber'] = pd.to_numeric(laps['LapNumber'], errors='coerce')
        laps = laps[laps['LapNumber'].notna()].copy()
        if laps.empty:
            continue

        laps['FuelLoad'] = race_max_laps - laps['LapNumber']

        if 'TyreLife' not in laps.columns:
            laps['TyreLife'] = np.nan
        laps['TyreLife'] = pd.to_numeric(laps['TyreLife'], errors='coerce')
        missing_tyre = laps['TyreLife'].isna()
        if missing_tyre.any():
            laps.loc[missing_tyre, 'TyreLife'] = (
                laps.loc[missing_tyre]
                .groupby(['Stint'], dropna=False)
                .cumcount() + 1
            )

        # Build CorrectedLapTime_Global using the class pipeline.
        laps = analyzer.add_drs_correction_to_laps(laps)
        laps = analyzer.add_dirty_air_correction_to_laps(laps)
        laps = analyzer.add_track_evolution_correction_to_laps(laps)
        laps = analyzer.add_temperature_correction_to_laps(laps)

        if 'CorrectedLapTime_Global' not in laps.columns:
            continue

        laps['CorrectedLapTime_Global'] = pd.to_numeric(laps['CorrectedLapTime_Global'], errors='coerce')
        laps = laps[laps['CorrectedLapTime_Global'].notna()].copy()
        if laps.empty:
            continue

        if 'TrackTemp_C' in laps.columns:
            laps['TrackTemp'] = pd.to_numeric(laps['TrackTemp_C'], errors='coerce')
        elif 'TrackTemp' in laps.columns:
            laps['TrackTemp'] = pd.to_numeric(laps['TrackTemp'], errors='coerce')
        else:
            laps['TrackTemp'] = np.nan

        laps['is_inlier_ransac'] = True
        for _, grp in laps.groupby(['Stint', 'Compound'], dropna=False):
            if len(grp) < 5:
                continue

            x = grp[['TyreLife']].to_numpy(dtype=float)
            y = grp['CorrectedLapTime_Global'].to_numpy(dtype=float)

            valid = np.isfinite(x).ravel() & np.isfinite(y)
            if valid.sum() < 5:
                continue

            x_valid = x[valid]
            y_valid = y[valid]
            idx_valid = grp.index[valid]

            try:
                ransac = RANSACRegressor(
                    estimator=LinearRegression(),
                    min_samples=max(3, int(0.5 * len(x_valid))),
                    residual_threshold=0.8,
                    random_state=42
                )
                ransac.fit(x_valid, y_valid)
                laps.loc[idx_valid, 'is_inlier_ransac'] = ransac.inlier_mask_
            except Exception:
                laps.loc[idx_valid, 'is_inlier_ransac'] = True

        laps = laps[laps['is_inlier_ransac']].copy()
        if laps.empty:
            continue

        best_corrected_global = laps['CorrectedLapTime_Global'].min()
        laps['DeltaToBest'] = laps['CorrectedLapTime_Global'] - best_corrected_global

        laps['Year'] = year
        laps['Event'] = event_name
        laps['Driver'] = drv
        laps['Abrasivity'] = abrasivity
        laps['LateralEnergy'] = lateral_energy
        laps['CompoundEncoded'] = laps['Compound'].map(COMPOUND_MAP)

        cols = [
            'Year', 'Event', 'Driver', 'Compound', 'CompoundEncoded', 'TyreLife',
            'TrackTemp', 'FuelLoad', 'Abrasivity', 'LateralEnergy', 'DeltaToBest',
            'LapNumber', 'Stint', 'CorrectedLapTime_Global'
        ]
        all_rows.append(laps[cols])

    if not all_rows:
        return pd.DataFrame()

    return pd.concat(all_rows, axis=0, ignore_index=True)


def build_master_dataset_from_cached(year=2024, target_rows=20000):
    events = get_cached_race_events(year=year)

    datasets = []
    for event in events:
        try:
            event_df = build_event_driver_laps(year=year, event_name=event, session_type='R')
            if not event_df.empty:
                datasets.append(event_df)
                print(f'[OK] {event}: {len(event_df)} lignes')
            else:
                print(f'[VIDE] {event}: 0 ligne')
        except Exception as exc:
            print(f'[ERREUR] {event}: {exc}')

    if not datasets:
        return pd.DataFrame(), events

    master = pd.concat(datasets, axis=0, ignore_index=True)

    master = master.replace([np.inf, -np.inf], np.nan).dropna(subset=[
        'CompoundEncoded', 'TyreLife', 'TrackTemp', 'FuelLoad',
        'Abrasivity', 'LateralEnergy', 'DeltaToBest', 'CorrectedLapTime_Global'
    ])

    if len(master) > target_rows:
        master = master.sample(n=target_rows, random_state=42).sort_values(['Year', 'Event', 'Driver', 'LapNumber'])
    else:
        master = master.sort_values(['Year', 'Event', 'Driver', 'LapNumber'])

    master = master.reset_index(drop=True)
    return master, events

In [ ]:
# Controle rapide de distribution par GP et pilote
if master_dataset.empty:
    print('Dataset vide')
else:
    display(master_dataset.groupby('Event').size().rename('rows_per_event').sort_values(ascending=False))
    display(master_dataset.groupby(['Event', 'Driver']).size().rename('rows_per_driver_event').head(30))

Event
Dutch Grand Prix             1274
Austrian Grand Prix          1206
Spanish Grand Prix           1131
Hungarian Grand Prix         1124
Emilia Romagna Grand Prix    1073
Mexico City Grand Prix       1011
Singapore Grand Prix         1004
Bahrain Grand Prix            977
Monaco Grand Prix             908
Miami Grand Prix              903
Italian Grand Prix            873
Abu Dhabi Grand Prix          855
Australian Grand Prix         811
Azerbaijan Grand Prix         761
Las Vegas Grand Prix          752
Chinese Grand Prix            731
Saudi Arabian Grand Prix      712
Belgian Grand Prix            708
Japanese Grand Prix           707
Qatar Grand Prix              646
British Grand Prix            526
Canadian Grand Prix           199
dtype: int64

Event                  Driver
Abu Dhabi Grand Prix   1         52
                       10        48
                       14        48
                       16        48
                       18        46
                       20        44
                       22        48
                       23        47
                       24        46
                       27        50
                       30        44
                       4         51
                       43        19
                       44        49
                       55        51
                       61        45
                       63        49
                       77        24
                       81        46
Australian Grand Prix  1          3
                       10        45
                       11        48
                       14        50
                       16        49
                       18        48
                       20        45
                       22        4

In [ ]:
master_dataset, used_events = build_master_dataset_from_cached(year=2024, target_rows=20000)

print('\n=== Resume extraction ===')
print('Nombre de GP utilises (cache, hors United States GP):', len(used_events))
print('Nombre de lignes dataset:', len(master_dataset))
print('Colonnes:', master_dataset.columns.tolist())

display(master_dataset.head(20))

output_path = Path('classes') / 'master_dataset_partie2_2024.csv'
master_dataset.to_csv(output_path, index=False)
print('CSV exporte:', output_path)

C:\Users\user\AppData\Roaming\Python\Python313\site-packages\fastf1\core.py:3129: FutureWarning: pick_driver is deprecated and will be removed in a future release. Use pick_drivers instead.
  warnings.warn(("pick_driver is deprecated and will be removed"
C:\Users\user\AppData\Roaming\Python\Python313\site-packages\fastf1\core.py:3129: FutureWarning: pick_driver is deprecated and will be removed in a future release. Use pick_drivers instead.
  warnings.warn(("pick_driver is deprecated and will be removed"
C:\Users\user\AppData\Roaming\Python\Python313\site-packages\fastf1\core.py:3129: FutureWarning: pick_driver is deprecated and will be removed in a future release. Use pick_drivers instead.
  warnings.warn(("pick_driver is deprecated and will be removed"
C:\Users\user\AppData\Roaming\Python\Python313\site-packages\fastf1\core.py:3129: FutureWarning: pick_driver is deprecated and will be removed in a future release. Use pick_drivers instead.
  warnings.warn(("pick_driver is deprecated a

[OK] Bahrain Grand Prix: 977 lignes


C:\Users\user\AppData\Roaming\Python\Python313\site-packages\fastf1\core.py:3129: FutureWarning: pick_driver is deprecated and will be removed in a future release. Use pick_drivers instead.
  warnings.warn(("pick_driver is deprecated and will be removed"
C:\Users\user\AppData\Roaming\Python\Python313\site-packages\fastf1\core.py:3129: FutureWarning: pick_driver is deprecated and will be removed in a future release. Use pick_drivers instead.
  warnings.warn(("pick_driver is deprecated and will be removed"
C:\Users\user\AppData\Roaming\Python\Python313\site-packages\fastf1\core.py:3129: FutureWarning: pick_driver is deprecated and will be removed in a future release. Use pick_drivers instead.
  warnings.warn(("pick_driver is deprecated and will be removed"
C:\Users\user\AppData\Roaming\Python\Python313\site-packages\fastf1\core.py:3129: FutureWarning: pick_driver is deprecated and will be removed in a future release. Use pick_drivers instead.
  warnings.warn(("pick_driver is deprecated a

[OK] Saudi Arabian Grand Prix: 712 lignes


C:\Users\user\AppData\Roaming\Python\Python313\site-packages\fastf1\core.py:3129: FutureWarning: pick_driver is deprecated and will be removed in a future release. Use pick_drivers instead.
  warnings.warn(("pick_driver is deprecated and will be removed"
C:\Users\user\AppData\Roaming\Python\Python313\site-packages\fastf1\core.py:3129: FutureWarning: pick_driver is deprecated and will be removed in a future release. Use pick_drivers instead.
  warnings.warn(("pick_driver is deprecated and will be removed"
C:\Users\user\AppData\Roaming\Python\Python313\site-packages\fastf1\core.py:3129: FutureWarning: pick_driver is deprecated and will be removed in a future release. Use pick_drivers instead.
  warnings.warn(("pick_driver is deprecated and will be removed"
C:\Users\user\AppData\Roaming\Python\Python313\site-packages\fastf1\core.py:3129: FutureWarning: pick_driver is deprecated and will be removed in a future release. Use pick_drivers instead.
  warnings.warn(("pick_driver is deprecated a

...

C:\Users\user\AppData\Roaming\Python\Python313\site-packages\fastf1\core.py:3129: FutureWarning: pick_driver is deprecated and will be removed in a future release. Use pick_drivers instead.
  warnings.warn(("pick_driver is deprecated and will be removed"
C:\Users\user\AppData\Roaming\Python\Python313\site-packages\fastf1\core.py:3129: FutureWarning: pick_driver is deprecated and will be removed in a future release. Use pick_drivers instead.
  warnings.warn(("pick_driver is deprecated and will be removed"
C:\Users\user\AppData\Roaming\Python\Python313\site-packages\fastf1\core.py:3129: FutureWarning: pick_driver is deprecated and will be removed in a future release. Use pick_drivers instead.
  warnings.warn(("pick_driver is deprecated and will be removed"
C:\Users\user\AppData\Roaming\Python\Python313\site-packages\fastf1\core.py:3129: FutureWarning: pick_driver is deprecated and will be removed in a future release. Use pick_drivers instead.
  warnings.warn(("pick_driver is deprecated a

[OK] Abu Dhabi Grand Prix: 855 lignes

=== Resume extraction ===
Nombre de GP utilises (cache, hors United States GP): 23
Nombre de lignes dataset: 18892
Colonnes: ['Year', 'Event', 'Driver', 'Compound', 'CompoundEncoded', 'TyreLife', 'TrackTemp', 'FuelLoad', 'Abrasivity', 'LateralEnergy', 'DeltaToBest', 'LapNumber', 'Stint', 'CorrectedLapTime']


C:\Users\user\AppData\Roaming\Python\Python313\site-packages\fastf1\core.py:3129: FutureWarning: pick_driver is deprecated and will be removed in a future release. Use pick_drivers instead.
  warnings.warn(("pick_driver is deprecated and will be removed"
C:\Users\user\AppData\Roaming\Python\Python313\site-packages\fastf1\core.py:3129: FutureWarning: pick_driver is deprecated and will be removed in a future release. Use pick_drivers instead.
  warnings.warn(("pick_driver is deprecated and will be removed"
C:\Users\user\AppData\Roaming\Python\Python313\site-packages\fastf1\core.py:3129: FutureWarning: pick_driver is deprecated and will be removed in a future release. Use pick_drivers instead.
  warnings.warn(("pick_driver is deprecated and will be removed"


,Year,Event,Driver,Compound,CompoundEncoded,TyreLife,TrackTemp,FuelLoad,Abrasivity,LateralEnergy,DeltaToBest,LapNumber,Stint,CorrectedLapTime
0,2024,Abu Dhabi Grand Prix,1,MEDIUM,1,4.0,31.9,54.0,3,3,0.471,4.0,1.0,89.644
1,2024,Abu Dhabi Grand Prix,1,MEDIUM,1,5.0,31.7,53.0,3,3,0.815,5.0,1.0,89.988
2,2024,Abu Dhabi Grand Prix,1,MEDIUM,1,6.0,31.8,52.0,3,3,0.449,6.0,1.0,89.622
3,2024,Abu Dhabi Grand Prix,1,MEDIUM,1,7.0,31.8,51.0,3,3,0.957,7.0,1.0,90.130
4,2024,Abu Dhabi Grand Prix,1,MEDIUM,1,8.0,31.5,50.0,3,3,0.931,8.0,1.0,90.104
5,2024,Abu Dhabi Grand Prix,1,MEDIUM,1,9.0,31.6,49.0,3,3,0.587,9.0,1.0,89.760
6,2024,Abu Dhabi Grand Prix,1,MEDIUM,1,10.0,31.7,48.0,3,3,1.626,10.0,1.0,90.799
7,2024,Abu Dhabi Grand Prix,1,MEDIUM,1,11.0,31.2,47.0,3,3,1.607,11.0,1.0,90.780
8,2024,Abu Dhabi Grand Prix,1,MEDIUM,1,12.0,31.6,46.0,3,3,1.411,12.0,1.0,90.584
9,2024,Abu Dhabi Grand Prix,1,MEDIUM,1,13.0,31.6,45.0,3,3,1.460,13.0,1.0,90.633


CSV exporte: classes\master_dataset_partie2_2024.csv
